In [ ]:
# | default_exp egu26

# EGU26 talk plots

> Reproducible plot functions for slides 5, 6, 7 of the EGU26 oral talk ("Planet Four — Inter- and Intra-annual Variability of Dark Regolith on Ice Coverage at the Martian South Polar Region").

In [ ]:
# | export
"""egu26 — figure-generation helpers for the EGU26 talk slides 5/6/7.

Each function uses the generic plotting machinery from :mod:`p4tools.plotting`
(``apply_talk_context``, ``histogram_kde``, ``kde_per_group``,
``smallmult_highlight_grid``) plus the data joiners
:func:`p4tools.io.attach_my` and :func:`p4tools.io.attach_roi`. The per-tile
coverage input is computed on first call by
:func:`p4tools.production.coverage.compute_per_tile_coverage`, then served
from a parquet cache thereafter.
"""
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from p4tools.io import attach_my, attach_roi
from p4tools.plotting import (
    apply_talk_context,
    histogram_kde,
    kde_per_group,
    smallmult_highlight_grid,
)
from p4tools.production.coverage import compute_per_tile_coverage

SLIDE7_ROIS = ("Inca_City", "Ithaca")

DEFAULT_OUT_DIR = Path("figures/coverage/talk")


def _load_coverage(coverage: pd.DataFrame | None, version: str) -> pd.DataFrame:
    if coverage is not None:
        return coverage
    return compute_per_tile_coverage(version=version)


def _resolve_out(out: str | Path | None, default_name: str) -> Path:
    if out is None:
        DEFAULT_OUT_DIR.mkdir(parents=True, exist_ok=True)
        return DEFAULT_OUT_DIR / default_name
    p = Path(out)
    p.parent.mkdir(parents=True, exist_ok=True)
    return p


## Slide 5 — Overall coverage statistics

In [ ]:
# | export
def plot_slide_5(
    coverage: pd.DataFrame | None = None,
    *,
    version: str = "v3.1",
    out: str | Path | None = None,
) -> Path:
    """Slide 5: cap-wide per-tile coverage histogram + KDE (talk context)."""
    apply_talk_context()
    cov = _load_coverage(coverage, version)
    pct = cov["Coverage"].to_numpy() * 100
    fig, ax = plt.subplots(figsize=(12, 7), dpi=120)
    histogram_kde(pct, ax=ax)
    ax.set_xlabel("per-tile coverage  [%]")
    ax.set_ylabel("density")
    ax.set_title(f"v3.1 per-tile coverage distribution  (N = {len(pct):,} tiles, KDE overlay)")
    fig.tight_layout()
    out_path = _resolve_out(out, "coverage_histogram_talk.png")
    fig.savefig(out_path, bbox_inches="tight")
    plt.close(fig)
    return out_path


## Slide 6 — Inter-annual

In [ ]:
# | export
def plot_slide_6(
    coverage: pd.DataFrame | None = None,
    *,
    version: str = "v3.1",
    out: str | Path | None = None,
) -> Path:
    """Slide 6: inter-annual KDE overlay per Mars Year (talk context)."""
    apply_talk_context()
    cov = attach_my(_load_coverage(coverage, version), version=version)
    cov = cov.dropna(subset=["MY"]).copy()
    cov["MY"] = cov["MY"].astype(int)
    cov["Coverage_pct"] = cov["Coverage"] * 100

    def _label(my, sub):
        return f"MY {my}   N={len(sub):>6,}   med={float(np.median(sub)):5.2f}%"

    fig, ax = plt.subplots(figsize=(14, 8), dpi=120)
    kde_per_group(cov, group_col="MY", value_col="Coverage_pct", ax=ax,
                  label_fn=_label)
    ax.set_xlabel("per-tile coverage  [%]")
    ax.set_ylabel("density")
    ax.set_title("Fractional coverage per Mars Year — v3.1")
    legend = ax.legend(frameon=False, loc="upper right", prop={"family": "monospace"})
    legend.set_title("")
    fig.tight_layout()
    out_path = _resolve_out(out, "coverage_per_MY_overlay_talk.png")
    fig.savefig(out_path, bbox_inches="tight")
    plt.close(fig)
    return out_path


## Slide 7 — Inter-regional (one ROI per call)

In [ ]:
# | export
def plot_slide_7(
    roi: str,
    coverage: pd.DataFrame | None = None,
    *,
    version: str = "v3.1",
    out: str | Path | None = None,
) -> Path:
    """Slide 7: per-MY small-multiples of coverage vs L_s for one ROI (talk context)."""
    apply_talk_context()
    cov = _load_coverage(coverage, version)
    per_obsid = (
        cov.groupby("obsid")["Coverage"]
        .agg(median="median",
             q25=lambda s: s.quantile(0.25),
             q75=lambda s: s.quantile(0.75))
        .reset_index()
    )
    per_obsid = attach_roi(attach_my(per_obsid, version=version))
    sub = per_obsid[per_obsid["roi_name"] == roi].dropna(subset=["MY", "l_s"]).copy()
    if sub.empty:
        raise RuntimeError(f"no data for ROI {roi!r}")
    sub["MY"] = sub["MY"].astype(int)
    sub["median"] = sub["median"] * 100
    sub["q25"] = sub["q25"] * 100
    sub["q75"] = sub["q75"] * 100

    fig = smallmult_highlight_grid(
        sub, panel_col="MY", x_col="l_s", y_col="median",
        yerr=("q25", "q75"), ncols=3, x_lim=(155, 345),
        figsize=(14, 10),
    )
    for ax in fig.axes:
        if ax.get_visible():
            ax.set_xlabel("L$_s$  [°]")
            ax.set_ylabel("median tile coverage  [%]")
    fig.suptitle(f"{roi.replace('_', ' ')} — coverage vs L$_s$, per Mars Year (v3.1)")
    fig.tight_layout()
    out_path = _resolve_out(out, f"coverage_smallmult_{roi}_talk.png")
    fig.savefig(out_path, bbox_inches="tight")
    plt.close(fig)
    return out_path


In [ ]:
# | export
if __name__ == "__main__":
    plot_slide_5()
    plot_slide_6()
    for roi in SLIDE7_ROIS:
        plot_slide_7(roi)
